In [ ]:
!pip install optuna catboost koreanize-matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# 데이터 전처리(데이터 분리, 스케일링, 인코딩, 파이프라인(데이터 누수 방지))관련
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 모델 학습 및 성능평가 관련
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score # (y_true, y_pred, sample_weight(Default=None))

# Optuna 관련
import optuna
from optuna import Trial
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
    plot_contour
)
# Optuna 로깅 레벨 조정
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 랜덤 시드 고정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Optuna 버전: {optuna.__version__}")


Optuna 버전: 4.7.0


In [ ]:
# 데이터 로드 & 확인
calories_data = pd.read_csv("/content/final_train_adj.csv")
calories_data.columns = [c.strip() for c in calories_data.columns]    # 컬럼명 앞뒤 공백 제거

# 해당 csv의 컬럼명으로 변경 필요
calories_data = calories_data[[
    "ex_dura", "bpm", "body_temp", "hr_ratio", "age", "gender",
    "height_cm", "weight_kg", "bmi", "log_exercise_stress_index",
    "log_bsa_intensity_time", "calories_burned"
    ]].copy()

print(calories_data.shape)    # (7500, 11)
display(calories_data.info())
print(f"중복 확인: {calories_data.duplicated()}")

(7500, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7500 entries, 0 to 7499
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ex_dura                    7500 non-null   float64
 1   bpm                        7500 non-null   float64
 2   body_temp                  7500 non-null   float64
 3   hr_ratio                   7500 non-null   float64
 4   age                        7500 non-null   int64  
 5   gender                     7500 non-null   object 
 6   height_cm                  7500 non-null   float64
 7   weight_kg                  7500 non-null   float64
 8   bmi                        7500 non-null   float64
 9   log_exercise_stress_index  7500 non-null   float64
 10  log_bsa_intensity_time     7500 non-null   float64
 11  calories_burned            7500 non-null   float64
dtypes: float64(10), int64(1), object(1)
memory usage: 703.3+ KB


None

중복 확인: 0       False
1       False
2       False
3       False
4       False
        ...  
7495    False
7496    False
7497    False
7498    False
7499    False
Length: 7500, dtype: bool


In [ ]:
# X, y 분리
X = calories_data.drop(columns=['calories_burned']).copy()
y = calories_data['calories_burned'].copy()

# train, validation 분리
# 이상치 데이터 flag 정의
# 체온 >= 41 이고 bpm < 100.0 이면 outlier_flag=1
outlier_flag = ((calories_data["body_temp"] >= 41) & (calories_data["bpm"] < 100.0)).astype(int)

X_train, X_valid, y_train, y_valid, flag_train, flag_valid = train_test_split(
    X, y, outlier_flag,
    test_size=0.2,
    random_state=42,
    stratify=outlier_flag
)

# 분리된 형태 확인
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape, flag_train.shape, flag_valid.shape

((6000, 11), (1500, 11), (6000,), (1500,), (6000,), (1500,))

# Optuna로 최적화

In [ ]:
print("\n" + "="*60)
print("CatBoost Optuna 최적화")
print("="*60)

# 1) sample_weight: 기본 1.0, outlier만 0.5
def make_sample_weight_from_flag(flag, w_bad=0.5):
    w = np.ones(len(flag), dtype=float)
    w[np.asarray(flag) == 1] = w_bad
    return w

# 2) StratifiedKFold (회귀지만 flag 기준 stratify)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 1000, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
        "loss_function": "RMSE",
        "random_state": RANDOM_STATE,
        "verbose": 0
    }

    rmses = []

    # stratify 기준은 y가 아니라 flag_train
    for tr_idx, va_idx in skf.split(X_train, flag_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        # train fold에만 sample_weight 적용
        w_tr = make_sample_weight_from_flag(flag_train.iloc[tr_idx], w_bad=0.5)

        model = CatBoostRegressor(**params)
        cat_features = ["gender"]    # 범주형 컬럼 명시
        model.fit(X_tr, y_tr, sample_weight=w_tr, cat_features=cat_features)

        pred = model.predict(X_va)
        rmse = mean_squared_error(y_va, pred)
        rmses.append(rmse)

    return float(np.mean(rmses))

# Study 생성 및 최적화
study_cat = optuna.create_study(
    direction="minimize",
    study_name="catboost_full"
)

study_cat.optimize(objective_catboost, n_trials=100, show_progress_bar=True)

print("Best RMSE:", study_cat.best_value)
print("Best params:", study_cat.best_params)


CatBoost Optuna 최적화


  0%|          | 0/100 [00:00<?, ?it/s]

Best RMSE: 1.1051674944368033
Best params: {'iterations': 2648, 'learning_rate': 0.020904348295798084, 'depth': 7, 'l2_leaf_reg': 1.022821588384818, 'border_count': 169, 'bagging_temperature': 0.10252235943651293, 'random_strength': 5.909998928357005}


In [ ]:
# CatBoost 파라미터 중요도
fig = plot_param_importances(study_cat)
fig.update_layout(title="CatBoost - Parameter Importances", height=500)
fig.show()

# GridSearchCV 하이퍼파라미터 탐색

In [ ]:
# 하이퍼파라미터 그리드
param_grid = {
    "learning_rate": [0.020, 0.021, 0.025],
    "l2_leaf_reg": [0.5, 1.023, 1.5],
    "iterations": [2200, 2648, 3200],
    # 나머지는 Optuna best 고정
    "depth": [7],
    "border_count": [169],
    "bagging_temperature": [0.10],
    "random_strength": [5.9],
    }

print("파라미터 그리드:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\n총 조합 수: {total_combinations}개")


파라미터 그리드:
  learning_rate: [0.02, 0.021, 0.025]
  l2_leaf_reg: [0.5, 1.023, 1.5]
  iterations: [2200, 2648, 3200]
  depth: [7]
  border_count: [169]
  bagging_temperature: [0.1]
  random_strength: [5.9]

총 조합 수: 27개


In [ ]:
# GridSearchCV
print("\nTuning CatBoost Regressor...")
print("(교차검증 수행 중...)\n")

# 1) 전체 train에 대한 sample_weight (길이 = len(X_train))
def make_sample_weight_from_flag(flag, w_bad=0.5):
    w = np.ones(len(flag), dtype=float)
    w[np.asarray(flag) == 1] = w_bad
    return w

w_train_full = make_sample_weight_from_flag(flag_train, w_bad=0.5)

# 2) StratifiedKFold split을 flag 기준으로 "고정"해서 생성
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(skf.split(X_train, flag_train))

# 3) 기본 estimator (고정 파라미터)
cat_reg = CatBoostRegressor(
    loss_function="RMSE",
    random_state=RANDOM_STATE,
    verbose=0,
    # task_type="GPU",
    # devices="0"
)

grid_reg = GridSearchCV(
    estimator=cat_reg,
    param_grid=param_grid,
    cv=cv_splits,  # ✅ flag stratified splits
    scoring="neg_root_mean_squared_error",  # ✅ GridSearchCV는 neg가 기본
    n_jobs=-1,
    verbose=1
)

# 4) fit: sample_weight는 전체 길이로, cat_features도 fit 파라미터로 전달
cat_features = ["gender"]

grid_reg.fit(
    X_train, y_train,
    sample_weight=w_train_full,
    cat_features=cat_features
)

print("Best RMSE (CV):", -grid_reg.best_score_)
print("Best params:", grid_reg.best_params_)
best_reg = grid_reg.best_estimator_


Tuning CatBoost Regressor...
(교차검증 수행 중...)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best RMSE (CV): 1.0041735128077138
Best params: {'bagging_temperature': 0.1, 'border_count': 169, 'depth': 7, 'iterations': 3200, 'l2_leaf_reg': 0.5, 'learning_rate': 0.021, 'random_strength': 5.9}


In [ ]:
# 최적 모델 추출
best_reg = grid_reg.best_estimator_

print("[최적화 결과]")
print(f"Best RMSE Score (Train CV): {grid_reg.best_score_:.4f}")
print("\nBest Parameters:")
for param, value in grid_reg.best_params_.items():
    print(f"  {param}: {value}")


[최적화 결과]
Best RMSE Score (Train CV): -1.0042

Best Parameters:
  bagging_temperature: 0.1
  border_count: 169
  depth: 7
  iterations: 3200
  l2_leaf_reg: 0.5
  learning_rate: 0.021
  random_strength: 5.9


In [ ]:
# 상위 3개 결과
cv_results = pd.DataFrame(grid_reg.cv_results_)

# 보기 좋은 컬럼만 선택(존재하는 것만)
candidate_cols = [
    'param_iterations', 'param_learning_rate', 'param_depth',
    'param_l2_leaf_reg', 'param_border_count',
    'mean_test_score', 'std_test_score', 'rank_test_score'
]
cols = [c for c in candidate_cols if c in cv_results.columns]

top_results = cv_results.nsmallest(3, 'rank_test_score')[cols]

print("\n상위 3개 파라미터 조합:")
display(top_results)


In [ ]:
# 테스트 데이터 예측 및 평가
y_pred = best_reg.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print(f"valid RMSE: {rmse:.4f}")
print(f"valid R2 Score: {r2:.4f}")


valid RMSE: 0.9484
valid R2 Score: 0.9998


In [ ]:
# 과적합 여부 확인: CV RMSE는 좋은데 holdout RMSE가 확 튄다 → 과적합 가능성↑. 비슷하면 문제 없음
pred_valid = best_reg.predict(X_valid)
rmse_valid = np.sqrt(np.mean((y_valid - pred_valid)**2))
print("Holdout RMSE:", rmse_valid)

Holdout RMSE: 0.9483886238777314


# 피처 중요도 분석

**CatBoost 내장 시각화**

In [ ]:
# CatBoost 피처 중요도 (PredictionValuesChange 방식)
feature_importance = best_reg.get_feature_importance()
feature_names = X.columns

# 데이터프레임 생성
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(6, 4))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# 피처 중요도 추출
importances = best_reg.get_feature_importance()
feature_names = X.columns
indices = np.argsort(importances)[::-1]

importance_df = pd.DataFrame({
    'Feature': feature_names[indices],
    'Importance': importances[indices]
})

print("\n피처 중요도 순위:")
display(importance_df)

plt.figure(figsize=(6, 4))
sns.barplot(x=importance_df['Importance'], y=importance_df['Feature'])
plt.title("Feature Importances (CatBoost)")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()
